# RAIL lung-nodule counting — Colab setup

Before running anything:
1. **Runtime > Change runtime type > GPU** (T4 is fine).
2. Upload `datasets/LDIC-IDRI-subset/` (the DICOM data + `annotations.csv`, ~11GB) to your Google Drive — this notebook does not transfer it for you.
3. Add a Colab secret named `HF_TOKEN` (key icon in the left sidebar) holding a Hugging Face access token that has accepted the MedGemma license at https://huggingface.co/google/medgemma-1.5-4b-it.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Point this at wherever you uploaded `LDIC-IDRI-subset` under Drive:

In [ ]:
DATA_DIR = "/content/drive/MyDrive/rail-data/LDIC-IDRI-subset"  # <-- update to your actual path

import pathlib
assert pathlib.Path(DATA_DIR).exists(), f"{DATA_DIR} not found — check the path/upload"

In [ ]:
!git clone https://github.com/freya-gul/rail.git
%cd rail

In [ ]:
!pip install -q -r requirements.txt

pylidc keeps its own scan index (built by scanning the configured DICOM directory the first time it's queried) and reads its config from `~/.pylidcrc`, not the repo's copy — write one pointing at the Drive path:

In [ ]:
import pathlib

pylidcrc = pathlib.Path.home() / ".pylidcrc"
pylidcrc.write_text(f"[dicom]\npath = {DATA_DIR}/lidc_idri\nwarn = True\n")
print(pylidcrc.read_text())

In [ ]:
from google.colab import userdata
from huggingface_hub import login

login(token=userdata.get('HF_TOKEN'))

Sanity check: GPU visible to torch, and the detector/MedGemma scripts will now pick it (they already default to `cuda` > `mps` > `cpu`):

In [ ]:
import torch
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print(torch.cuda.get_device_name(0))

## Run the pipeline

`ground_truth_annotations.json` and `detection_cache/` are already committed to the repo for patients 1-20, so the two cells below only need to run if you want to regenerate them or extend the patient range.

In [ ]:
# Ground truth (pylidc) — CPU-only, only needed to regenerate/extend beyond patients 1-20
!python image_download/6_ground_truth_annotations.py --start 1 --end 20

In [ ]:
# MONAI RetinaNet detection + FROC evaluation vs. ground truth (GPU)
!python image_download/7_evaluate_detection.py --start 1 --end 20

In [ ]:
# MedGemma-only whole-volume counting baseline vs. ground truth (GPU)
!python image_download/8_evaluate_medgemma_counting.py --start 1 --end 20

Results land back in the repo: `image_download/detection_froc.csv`, `image_download/detection_counts.json`, `image_download/medgemma_counting_comparison.csv`. Both eval scripts cache per-patient work (`detection_cache/`, `medgemma_count_cache/`) so a run can be safely interrupted and resumed.